1. Setup & Load Final Model

In [4]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from utils import load_data
from features import compute_features
from models import train_model, predict_pass_probabilities

# 1. Load Data
X_train, y_train, X_test = load_data()

# 2. Re-Load your processed training features (to retrain the model on FULL data)
df_train_features = pd.read_csv('../data/processed/train_features.csv')

# 3. Train Final Model on ALL training data (No validation split this time!)
print("Training final model on all 8686 samples...")
X_full = df_train_features.drop(columns=['pass_id', 'target', 'candidate_id', 'sender_id'])
y_full = df_train_features['target']
final_model = train_model(X_full, y_full)

Loading data from c:\Users\ANKRI\Desktop\Q1\ELEN0062-1\Project3\Football-pass-prediction\data\raw...
Training final model on all 8686 samples...
Training Gradient Boosting with Calibration...


2. Process the Test Set

In [6]:
# 1. Load Raw Test Data
print("Loading raw test set...")
X_test_raw = pd.read_csv('../data/raw/input_test_set.csv') # Adjust path if needed (e.g. 'input_test_set.csv')

# 2. Compute Features (Force execution, ignore cache)
print(f"Computing NEW features (Packing, Blockage...) for {len(X_test_raw)} passes...")
df_test_features = compute_features(X_test_raw)

# 3. Save to disk (Overwriting the old file)
test_feat_path = '../data/processed/test_features.csv'
# Ensure directory exists
os.makedirs(os.path.dirname(test_feat_path), exist_ok=True)
df_test_features.to_csv(test_feat_path, index=False)
print(f"Saved fresh test features to {test_feat_path}")

# 4. Verification
# Print columns to confirm 'feat_packing' is there
print("Columns in test set:", df_test_features.columns.tolist())

Loading raw test set...
Computing NEW features (Packing, Blockage...) for 3000 passes...
Generating PRO features (Blockage, Congestion) for 3000 passes...
Saved fresh test features to ../data/processed/test_features.csv
Columns in test set: ['pass_id', 'candidate_id', 'sender_id', 'feat_dist', 'feat_angle', 'feat_team', 'feat_fwd', 'feat_pressure', 'feat_congestion', 'feat_blockage', 'feat_packing', 'rank_dist', 'rank_fwd', 'rank_pressure', 'rank_blockage']


3. Predict on Test Set

In [7]:
print("Predicting probabilities for test set...")

# 1. Define the EXACT feature list used for training
# These must match the columns generated in your compute_features function
features_used = [
    'feat_dist', 'feat_angle', 'feat_team', 'feat_fwd', 
    'feat_pressure', 'feat_congestion', 'feat_blockage', 'feat_packing',
    'rank_dist', 'rank_fwd', 'rank_pressure', 'rank_blockage'
]

# 2. Select strictly these columns from the test set
# This prevents the ValueError by ensuring input shape matches the model
X_test_model = df_test_features[features_used]

# 3. Predict Raw Probabilities (Class 1 = Receiver)
# We get a 1D array of 66,000 probabilities (3000 passes * 22 players)
raw_probas = final_model.predict_proba(X_test_model)[:, 1]

# 4. Reshape to (n_passes, 22) matrix
# Assumptions: 3000 test passes, data sorted by pass_id then candidate_id
n_passes = 3000
probas_matrix = raw_probas.reshape(n_passes, 22)

# 5. Normalize Probabilities (Critical for Brier Score)
# Raw probabilities for 22 candidates might sum to >1 or <1. We normalize them.
row_sums = probas_matrix.sum(axis=1, keepdims=True)
test_probas_normalized = probas_matrix / row_sums

# 6. Create DataFrame for inspection
# Columns P_1 to P_22
col_names = [f'P_{i}' for i in range(1, 23)]
test_probas = pd.DataFrame(test_probas_normalized, columns=col_names)

# Add Pass ID for reference
# (Taking every 22nd value from the original ID column)
test_pass_ids = df_test_features['pass_id'].values[::22]
test_probas.insert(0, 'Id', test_pass_ids)

print("Predictions ready.")
print(test_probas.head())

Predicting probabilities for test set...
Predictions ready.
   Id       P_1       P_2       P_3       P_4       P_5       P_6       P_7  \
0   0  0.020175  0.025844  0.006413  0.036371  0.052643  0.002611  0.013777   
1   1  0.007341  0.254411  0.001468  0.101000  0.008113  0.344709  0.000862   
2   2  0.017272  0.019428  0.005098  0.022454  0.010765  0.019428  0.037422   
3   3  0.145896  0.005268  0.003950  0.008247  0.004919  0.004342  0.009003   
4   4  0.047297  0.005874  0.049519  0.022387  0.006905  0.013699  0.023873   

        P_8       P_9  ...      P_13      P_14      P_15      P_16      P_17  \
0  0.016531  0.002561  ...  0.076525  0.001820  0.020043  0.259852  0.079264   
1  0.009429  0.016456  ...  0.010045  0.004148  0.006677  0.028867  0.001752   
2  0.010765  0.015388  ...  0.008543  0.027942  0.000190  0.021420  0.088262   
3  0.003534  0.001734  ...  0.000132  0.044542  0.005667  0.051866  0.272128   
4  0.002778  0.011825  ...  0.398105  0.182232  0.015761  0.05000

4. Format the Submission File

In [ ]:
import numpy as np
import os

def create_submission_file_simple(probas_df, estimated_acc, filename="submission.csv"):
    """
    Writes the submission file from the ALREADY RESHAPED dataframe.
    Input: probas_df with columns ['Id', 'P_1', 'P_2', ..., 'P_22']
    """
    print(f"Writing submission... Estimated Acc: {estimated_acc}")
    
    # Ensure directory exists
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    with open(filename, 'w') as f:
        # 1. Write Header (Strict Format: "Id","Predicted","P_1",...)
        header_cols = ['"Id"', '"Predicted"'] + [f'"P_{i}"' for i in range(1, 23)]
        f.write(",".join(header_cols) + "\n")
        
        # 2. Write Estimation Line (Strict Format: "Estimation",<acc>,0,0...)
        est_line = [f'"Estimation"', str(estimated_acc)] + ["0"] * 22
        f.write(",".join(est_line) + "\n")
        
        # 3. Write Prediction Rows
        # Iterate over the DataFrame rows
        for index, row in probas_df.iterrows():
            pass_id = int(row['Id'])
            
            # Extract probabilities P_1 to P_22
            # (Assuming columns 1 to 22 contain the probs. 
            # Column 0 is 'Id'. So values[1:] correspond to P_1..P_22)
            # SAFE WAY: Select by column names to be sure
            probs = row[[f'P_{i}' for i in range(1, 23)]].values.astype(float)
            
            # Determine Predicted Class (argmax + 1)
            predicted_player = np.argmax(probs) + 1
            
            # Format probabilities to avoid scientific notation
            probs_str = [f"{p:.6f}" for p in probs]
            
            # Construct Line
            line = [str(pass_id), str(predicted_player)] + probs_str
            f.write(",".join(line) + "\n")
            
    print(f"✅ File '{filename}' created successfully.")

# --- EXECUTE ---

# Note: Ensure test_probas is the dataframe from the PREVIOUS cell
# (The one with 'Id', 'P_1', 'P_2'...)
create_submission_file_simple(test_probas, my_estimated_acc, filename="../submissions/submission_fixed.csv")

Writing submission... Estimated Acc: 0.55
✅ File '../submissions/submission_fixed.csv' created successfully.
